This notebook uses the following data sources:
1. GHSL for high-resolution population data
2. Kummu et al. (v4) for high-resolution GDP (PPP) data
3. Earth Observation Group for high-resolution nighttime light intensity

The following sources are used to verify and harmonise data:
1. GHSL population data is verified against adm0 totals from SSP historical reference data
2. Re-distributed Kummu et al. GDP data is verified against adm0 totals from SSP historical reference data

The harmonised base year socioeconomic indicators are saved as GeoDataFrames to be used as the basis for method validation and future projections.

## 1. Importing required packages

The following cell needs to be run first whenever the kernel is restarted.

In [ ]:
from geocube.vector import vectorize

import geopandas as gpd

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm

import numpy as np

import pandas as pd

import rasterio
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.plot import show
from rasterio.features import rasterize
from rasterio.transform import from_origin

import rioxarray

from rtree import index

from scipy.optimize import curve_fit

from shapely.ops import nearest_points
from shapely.geometry import box

from sklearn.metrics import r2_score

import mapclassify

## 2. Country list

The following cell is used to define the list of countries under investigation. Their corresponding socioeconomic and electricity-related indicators are extracted from a pre-prepared dataset combining data from the SSP Scenario Explorer, the World Bank, and the IEA Data Browser.

In [ ]:
# Load the SSP dataset
df = pd.read_excel('input/socioeconomic_data_ssp_iea_2015.xlsx')

# User-defined representative countries list, based on lower-case 3-letter country code
representative_countries = df[(df['Code'] == 'nam')]
representative_countries['Code'] = representative_countries['Code'].str.upper()
representative_countries

## 3. Population data

### 3.1 Vectorising population raster and storing it as a GeoDataFrame

In [ ]:
for idx, country in representative_countries.iterrows():
    pop_file_name = country['Code'] + '_GHS_popcount_2015_30arcsec.tif'
    pop_raster_name = country['Code'] + '_pop_raster'
    pop_gdf_name = country['Code'] + '_pop_gdf'
    # opening and saving raster population data
    locals()[pop_raster_name] = rioxarray.open_rasterio('input/'+pop_file_name).squeeze().astype('float32')
    
    # naming raster layer data, corresponds to column name upon vectorisation
    locals()[pop_raster_name].name = '2015_POP'
    
    # vectorising and saving population data
    locals()[pop_gdf_name] = vectorize(locals()[pop_raster_name])

In [ ]:
for idx, country in representative_countries.iterrows():
    pop_gdf_name = country['Code'] + '_pop_gdf'
    print('Population of ' + country['Country'] + ':', f"{locals()[pop_gdf_name]['2015_POP'].sum():,.0f}")

### 3.2 Filtering out cells with less than 1 capita/km2

The purpose of this step is to remove noise from the original population dataset by filtering out cells with extremely low population density, i.e. less than 1 capita/km2. This threshold can be adjusted or this step could be skipped entirely if the resulting population total is significantly different from the original dataset. However, it is advised to at least round population count per cell to the nearest whole number to avoid distortion in later calculations, like GDP per capita.

In [ ]:
for idx, country in representative_countries.iterrows():
    pop_gdf_name = country['Code'] + '_pop_gdf'
    pop_gdf_gt1_name = country['Code'] + '_pop_gdf_gt1'
    print(f"\033[1m{country['Country']}\033[0m")
    # checking initial cell and population counts
    n_cells = len(locals()[pop_gdf_name]) # number of populated cells
    tot_pop = locals()[pop_gdf_name]['2015_POP'].sum() # population total
    print('total number of cells: ', n_cells)
    print('total population: ', f'{tot_pop:,.0f}')
    
    # creating new gdf including only cells with at least 1 cap/km2
    locals()[pop_gdf_gt1_name] = locals()[pop_gdf_name][locals()[pop_gdf_name]['2015_POP'] >= 1]
    n_cells_gt1 = len(locals()[pop_gdf_gt1_name]) # number of cells after filtering
    tot_pop_gt1 = locals()[pop_gdf_gt1_name]['2015_POP'].sum() # total population after filtering
    print('number of cells with at least 1 cap/km2: ', n_cells_gt1)
    print('total population (>= 1 cap/km2): ', f'{tot_pop_gt1:,.0f}')
    
    # checking filtered cell and population counts compared to original dataset
    print('cell ratio: ', f'{n_cells_gt1 / n_cells * 100:.1f} %')
    print('population ratio: ', f'{tot_pop_gt1 / tot_pop * 100:.1f} %')
    
    #pop_gdf_gt1.sort_values('2015_POP').head()

### 3.3 Validation and correction

The purpose of this step is to validate the base year population raster against official figures from the SSP Scenario Explorer historical reference values. A correction factor is calculated and applied uniformly to all populated cells based on the discrepancy in total population count.

In [ ]:
for idx, country in representative_countries.iterrows():
    pop_gdf_gt1_name = country['Code'] + '_pop_gdf_gt1'
    print(f"\033[1m{country['Country']}\033[0m")
    
    # calculating correction factor to align with SSP figures
    ssp_pop = df.loc[df['Country'] == country['Country'], 'POP'].values[0] # official figure from SSP historical reference (IIASA-WiC POP 2023)
    tot_pop = locals()[pop_gdf_gt1_name]['2015_POP'].sum() # population total from GHS
    corr_fac = ssp_pop / tot_pop
    print('total population, SSP:', f"{ssp_pop:,.0f}")
    print('total population, GHS:', f'{tot_pop:,.0f}')
    print('correction factor:', f"{corr_fac:.3f}")

    # applying correction factor
    locals()[pop_gdf_gt1_name]['2015_POP'] = locals()[pop_gdf_gt1_name]['2015_POP'] * corr_fac
    corr_pop = locals()[pop_gdf_gt1_name]['2015_POP'].sum()
    print('corrected total population:', f"{corr_pop:,.0f}")

## 4. Electrification status

### 4.1 Importing and preparing nighttime lights raster layer

This step includes importing the original nighttime lights layer, and downsampling it to match the resolution of the population and GDP datasets.

In [ ]:
# Downsampling nighttime lights raster layer to match the resolution of population and GDP layers, from 15 arcsec to 30 arcsec

# input and output raster files paths
# insert file name and extension if file is already uploaded to the same workspace, otherwise insert full file path
# it is recommended to upload a copy of all required raw files to the active workspace
for idx, country in representative_countries.iterrows():
    
    input_raster_path = 'input/' + country['Code'] + '_nighttime_lights_2015.tif' # source file
    output_raster_path = 'input/' + country['Code'] + '_nighttime_lights_2015_30arcsec.tif' # destination file
    
    # opening the input raster file
    with rasterio.open(input_raster_path) as src:
        # reading the metadata of the input raster
        src_transform = src.transform
        src_crs = src.crs
        src_width = src.width
        src_height = src.height
        src_count = src.count
    
        # defining the downsampling factor
        downsampling_factor = 30 / 15 # target resolution/original resolution in CRS units
    
        # calculating the new dimensions based on downsampling factor
        dst_width = src_width // downsampling_factor
        dst_height = src_height // downsampling_factor
    
        # calculating the new transformation
        dst_transform = src_transform * src_transform.scale(
            (src_width / dst_width),
            (src_height / dst_height)
        )
    
        # defining the metadata of the output raster
        dst_meta = src.meta.copy()
        dst_meta.update({
            'width': dst_width,
            'height': dst_height,
            'transform': dst_transform,
            'nodata': -999
        })
    
        # opening the output raster file
        with rasterio.open(output_raster_path, 'w', **dst_meta) as dst:
            # looping over all bands and resampling each as required
            # the resampling method used here is "max"
            # other resampling methods can be applied, like nearest, bilinear, etc.
            for i in range(1, src_count + 1):
                reproject(
                    source=rasterio.band(src, i),
                    destination=rasterio.band(dst, i),
                    src_transform=src_transform,
                    src_crs=src_crs,
                    dst_transform=dst_transform,
                    dst_crs=src_crs,
                    resampling=Resampling.max
                )
    
    # printing downsampling confirmation
    print(f"Downsampled raster saved to '{output_raster_path}'")

### 4.2 Determining electrification status of each populated cell

After the nighttime lights layer has been prepared for compatibility, it can be used to determine the initial electrification status of each populated cell. This step involves sampling the nighttime lights value corresponding to each populated cell. The minimum threshold of nighttime lights required to classify a populated cell as electrified is then iteratively calibrated such that the share of electrified population matches national statistics.

In [ ]:
# Defining a new function for raster sampling
def r_sample(raster_path, band, gdf, col_name):
    '''
    This function samples values from a raster layer corresponding to geometry centroids of a polygon GeoDataFrame.
    Inputs:
        raster_path (str): name/full path of raster file from which data will be sampled
        band (int): number of raster band from which values should be sampled
        gdf (var): variable name of the GeoDataFrame containing polygon geometries for which data needs to be sampled
        col_name (str): name to be given to the column containing sampled values in the GeoDataFrame
    Returns:
        Input GeoDataFrame with newly added column containing sampled values.
    '''
    # opening and reading data from the source raster file
    with rasterio.open(raster_path) as src:
        raster_data = src.read(band)
        raster_transform = src.transform
        raster_nodata = src.nodata

        # calculating centroids of the GeoDataFrame polygon geometries
        gdf['centroid'] = gdf.geometry.centroid

        # extracting raster values using polygon centroids
        centroid_coords = [(x, y) for x, y in zip(gdf['centroid'].x, gdf['centroid'].y)]
        raster_values = [val[0] for val in src.sample(centroid_coords, indexes=band)]

        # adding sampled raster values to the GeoDataFrame as a new column
        gdf[col_name] = raster_values
        return gdf

In [ ]:
# Sampling nighttime lights layer for each populated cell
for idx, country in representative_countries.iterrows():
    pop_gdf_gt1_name = country['Code'] + '_pop_gdf_gt1'
    pop_ntl_gdf_name = country['Code'] + '_pop_ntl_gdf'
    ntl_raster_name = 'input/' + country['Code'] + '_nighttime_lights_2015_30arcsec.tif'
    locals()[pop_ntl_gdf_name] = r_sample(ntl_raster_name, 1, locals()[pop_gdf_gt1_name], '2015_NTL')

NAM_pop_ntl_gdf.head()

In [ ]:
for idx, country in representative_countries.iterrows():
    pop_ntl_gdf_name = country['Code'] + '_pop_ntl_gdf'
    
    # Official electrification statistics
    # from World Bank: https://data.worldbank.org/indicator/EG.ELC.ACCS.ZS
    elec_access = df.loc[df['Country'] == country['Country'], 'Elec_Access'].values[0]
    
    # total population, both electrified and unelectrified
    tot_pop = locals()[pop_ntl_gdf_name]['2015_POP'].sum()
    
    # calibrating nighttime lights threshold to official electricity access rate to determine electrification status of each populated cell
    init_threshold = 0 # initial nighttime lights min. threshold
    tol = 0.05 / 100 # error tolerance in %

    if elec_access == 1:
        locals()[pop_ntl_gdf_name]['elec_stat'] = True
        print('Status report for ' + country['Country'])
        print('All populated cells are classified as electrified since electricity access rate is equal to ' + f"{elec_access:.1f}")
    else:   
        # classifying populated cells based on initial nighttime lights threshold
        locals()[pop_ntl_gdf_name]['elec_stat'] = np.where(locals()[pop_ntl_gdf_name]['2015_NTL'] > init_threshold, True, False)
        
        # calculating initial electrification share from total population
        elec_access_calc = locals()[pop_ntl_gdf_name].loc[locals()[pop_ntl_gdf_name]['elec_stat'] == True, '2015_POP'].sum() / tot_pop
        # calculating absolute error between initial calculation and official stats
        abs_err = elec_access_calc - elec_access
        
        # calibrating nighttime lights threshold based on defined error tolerance
        # case 1: too few electrified cells based on initial threshold
        if abs_err < 0:
            print('Status report for ' + country['Country'])
            print('Calculated electricity access rate: ', f'{(elec_access_calc * 100):.3f}', '%')
            print('Current nighttime lights threshold: ', init_threshold)
            print('Absolute error: ', f'{(abs_err * 100):.2f}', '%')
            print('Starting electricity access rate is below official figures, try adjusting the initial threshold to a lower value.')
        
        # case 2: acceptable electrification share based on initial threshold
        elif abs_err <= tol:
            print('Status report for ' + country['Country'])
            print('Calculated electricity access rate: ', f'{(elec_access_calc * 100):.3f}', '%')
            print('Current nighttime lights threshold: ', init_threshold)
            print('Absolute error: ', f'{(abs_err * 100):.2f}', '%')
            print('Resulting electricity access rate is within an acceptable error margin.')
        
        # case 3: too many electrified cells based on initial threshold
        else:
            adj = 0.5 # threshold adjustment step, can be changed
            adj_threshold = init_threshold + adj
            
            for i in range(1,50): # limited to 50 iterations, can be changed
                # re-calculating electrification share based on adjusted threshold
                locals()[pop_ntl_gdf_name]['elec_stat'] = np.where(locals()[pop_ntl_gdf_name]['2015_NTL'] > adj_threshold, True, False) # re-classify
                elec_access_calc = locals()[pop_ntl_gdf_name].loc[locals()[pop_ntl_gdf_name]['elec_stat'] == True, '2015_POP'].sum() / tot_pop # re-calculate share
                abs_err = elec_access_calc - elec_access # re-calculate error
                
                if abs_err < 0: # case 1
                    adj = adj/2
                    adj_threshold -= adj
                elif abs_err > tol: # case 3
                    adj_threshold += adj
                else: # case 2
                    break
            
            # final calculations after satisfying defined error tolerance or reaching 50 iterations 
            print('Status report for ' + country['Country'])
            print('Calculated electricity access rate: ', f'{(elec_access_calc * 100):.3f}', '%')
            print('Current nighttime lights threshold: ', adj_threshold)
            print('Absolute error: ', f'{(abs_err * 100):.2f}', '%')
            
            if abs_err <= tol:
                print('Resulting electricity access rate is within an acceptable error margin.')
            else:
                print('Error in electricity access rate is unacceptable, try adjusting the initial threshold.')     

## 5. Base year socio-economic indicators

This part involves various steps. First, total GDP data is obtained for each populated cell via raster sampling. However, some populated cells may not have corresponding GDP data. Thus, the population dataset is split into two; one for cells with available GDP data, and the other for cells with no available GDP data. For the first subset, the income level (i.e. GDP per capita) is calculated for each cell via dividing cell GDP by cell population. For the second subset, the income level is estimated based on the income level of the nearest cell with the same electrification status and available GDP data.

### 5.1 Obtaining GDP PPP data for each populated cell via raster sampling

In [ ]:
# sampling GDP values for each populated cell
for idx, country in representative_countries.iterrows():
    pop_gdp_gdf_name = country['Code'] + '_pop_gdp_gdf'
    pop_ntl_gdf_name = country ['Code'] + '_pop_ntl_gdf'
    
    gdp_raster_name = 'input/' + country['Code'] + '_gdpTot_1990_2020_30arcsec_Kummu_v4.tif' # GDP PPP in 2017 international USD
    
    locals()[pop_gdp_gdf_name] = r_sample(gdp_raster_name, 6, locals()[pop_ntl_gdf_name], '2015_GDP') #2015 values are in band 6
    locals()[pop_gdp_gdf_name]['2015_INC'] = locals()[pop_gdp_gdf_name]['2015_GDP'] / locals()[pop_gdp_gdf_name]['2015_POP']

NAM_pop_gdp_gdf.head()

In [ ]:
# preliminarily checking total GDP and verifying against SSP historical reference figures
for idx, country in representative_countries.iterrows():
    pop_gdp_gdf_name = country['Code'] + '_pop_gdp_gdf'
    ssp_gdp = df.loc[df['Country'] == country['Country'], 'GDP_2017USD'].values[0] # from SSP historical reference (OECD ENV-Growth 2023)
    tot_gdp = locals()[pop_gdp_gdf_name]['2015_GDP'].sum() # GDP total from Kummu et al. v4
    print('Checking total GDP of ' + country['Country'])
    print('Total GDP, SSP:', f"{ssp_gdp:,.0f}", 'USD')
    print('Total GDP, Kummu:', f"{tot_gdp:,.0f}", 'USD')
    print('Relative error:', f"{(tot_gdp - ssp_gdp) / ssp_gdp * 100:.2f}", '%')

### 5.2 Splitting population and GDP GeoDataFrame based on GDP data availability and electrification status

In [ ]:
for idx, country in representative_countries.iterrows():
    pop_gdp_gdf_name = country['Code'] + '_pop_gdp_gdf'
    pop_gdp_data_name = country['Code'] + '_pop_gdp_data'
    elec_pop_gdp_data_name = country['Code'] + '_elec_pop_gdp_data'
    unelec_pop_gdp_data_name = country['Code'] + '_unelec_pop_gdp_data'
    pop_gdp_nodata_name = country['Code'] + '_pop_gdp_nodata'
    elec_pop_gdp_nodata_name = country['Code'] + '_elec_pop_gdp_nodata'
    unelec_pop_gdp_nodata_name = country['Code'] + '_unelec_pop_gdp_nodata'

    print('GDP data availability in ' + country['Country'])
    # GeoDataFrame of populated cells with available GDP data
    locals()[pop_gdp_data_name] = locals()[pop_gdp_gdf_name][locals()[pop_gdp_gdf_name]['2015_GDP'] > 0]
    pop_gdp = locals()[pop_gdp_data_name]['2015_POP'].sum()
    print('Population with available GDP data:', f"{pop_gdp:,.0f}")
    
    # Further splitting by electrification status
    locals()[elec_pop_gdp_data_name] = locals()[pop_gdp_data_name][locals()[pop_gdp_data_name]['elec_stat'] == True]
    locals()[unelec_pop_gdp_data_name] = locals()[pop_gdp_data_name][locals()[pop_gdp_data_name]['elec_stat'] == False]
    
    # GeoDataFrame of populated cells with no available GDP data
    locals()[pop_gdp_nodata_name] = locals()[pop_gdp_gdf_name][locals()[pop_gdp_gdf_name]['2015_GDP'] == 0]
    pop_nogdp = locals()[pop_gdp_nodata_name]['2015_POP'].sum()
    print('Population with missing GDP data:', f"{pop_nogdp:,.0f}")
    print('Share of population with missing GDP data:', f"{pop_nogdp / (pop_nogdp + pop_gdp) * 100:.2f}", '%')
    
    # Further splitting by electrification status
    locals()[elec_pop_gdp_nodata_name] = locals()[pop_gdp_nodata_name][locals()[pop_gdp_nodata_name]['elec_stat'] == True]
    locals()[elec_pop_gdp_nodata_name] = locals()[elec_pop_gdp_nodata_name].drop(['2015_INC'], axis=1)
    locals()[unelec_pop_gdp_nodata_name] = locals()[pop_gdp_nodata_name][locals()[pop_gdp_nodata_name]['elec_stat'] == False]
    locals()[unelec_pop_gdp_nodata_name] = locals()[unelec_pop_gdp_nodata_name].drop(['2015_INC'], axis=1)
    
    # Double-check that split GDFs match total length of the original one
    print('length of full GDF:', len(locals()[pop_gdp_gdf_name]))
    print('combined length of the two split GDFs:', len(locals()[pop_gdp_data_name]) + len(locals()[pop_gdp_nodata_name]))
    print('combined length of the four split GDFs:', 
          len(locals()[elec_pop_gdp_data_name]) + len(locals()[unelec_pop_gdp_data_name]) + len(locals()[elec_pop_gdp_nodata_name]) + len(locals()[unelec_pop_gdp_nodata_name]))

In [ ]:
# Preliminary GDP summary statistics, broken down by electrification status

# printing disclaimer about currency unit, change as needed
print('All values are in constant international 2017 USD.')

for idx, country in representative_countries.iterrows():
    elec_pop_gdp_data_name = country['Code'] + '_elec_pop_gdp_data'
    unelec_pop_gdp_data_name = country['Code'] + '_unelec_pop_gdp_data'

    print('GDP statistics for ' + country['Country'])
    
    # GDP per capita is referred to as income level, abbv. INC   
    # checking general statistics for computed income level for electrified cells
    elec_max_inc = locals()[elec_pop_gdp_data_name]['2015_INC'].max()
    elec_min_inc = locals()[elec_pop_gdp_data_name]['2015_INC'].min()
    elec_avg_inc = locals()[elec_pop_gdp_data_name]['2015_INC'].mean()
    elec_w_avg_inc = locals()[elec_pop_gdp_data_name]['2015_GDP'].sum() / locals()[elec_pop_gdp_data_name]['2015_POP'].sum()
    print('Max. income level of electrified population: ', f'{elec_max_inc:,.0f}')
    print('Min. income level of electrified population: ', f'{elec_min_inc:,.0f}')
    print('Avg. income level of electrified population: ', f'{elec_avg_inc:,.0f}') # not weighted
    print('Weighted avg. income level of electrified population: ', f'{elec_w_avg_inc:,.0f}')
    
    # checking general statistics for computed income level for un-electrified cells
    unelec_max_inc = locals()[unelec_pop_gdp_data_name]['2015_INC'].max()
    unelec_min_inc = locals()[unelec_pop_gdp_data_name]['2015_INC'].min()
    unelec_avg_inc = locals()[unelec_pop_gdp_data_name]['2015_INC'].mean()
    unelec_w_avg_inc = locals()[unelec_pop_gdp_data_name]['2015_GDP'].sum() / locals()[unelec_pop_gdp_data_name]['2015_POP'].sum()
    print('Max. income level of unelectrified population: ', f'{unelec_max_inc:,.0f}')
    print('Min. income level of unelectrified population: ', f'{unelec_min_inc:,.0f}')
    print('Avg. income level of unelectrified population: ', f'{unelec_avg_inc:,.0f}')
    print('Weighted avg. income level of unelectrified population: ', f'{unelec_w_avg_inc:,.0f}')

### 5.3 Estimating GDP per capita for populated cells with no data based on nearest neighbour

In [ ]:
# Defining a new function for determining the nearest neighbour between two datasets
def nearest_inc(nodata_gdf, data_gdf):
    '''
    This function estimates income level for populated cells with no available GDP data based on the nearest neighbour from another dataset of populated cells with available GDP data.
    Inputs:
        nodata_gdf (var): GeoDataFrame of populated cells with no available GDP data
        data_gdf (var): GeoDataFrame of populated cells with available GDP data to look for nearest neighbours in
    Returns:
        nodata_gdf with two new columns; nearest neighbour index from data_gdf and corresponding estimated income level
    '''
    # ensuring both GeoDataFrames have the same CRS
    nodata_gdf = nodata_gdf.to_crs("EPSG:4326")
    data_gdf = data_gdf.to_crs("EPSG:4326")

    # creating a spatial index for data_gdf
    spatial_index = index.Index()
    for idx, geometry in data_gdf.geometry.items():
        spatial_index.insert(idx, geometry.bounds)

    # sub-function to find the spatial index of the nearest neighbor
    def find_nearest(geom):
        nearest_idx = list(spatial_index.nearest(geom.bounds, 1))[0]
        return nearest_idx

    # applying the nearest neighbor sub-function
    # adds a new column with the spatial index of the nearest neighbour with available data
    nodata_gdf['nearest_index'] = nodata_gdf.geometry.apply(find_nearest)

    # joining the data from data_gdf to nodata_gdf based on the nearest_index
    nodata_gdf = nodata_gdf.join(data_gdf, on='nearest_index', rsuffix='_nearest')
    
    # dropping extra columns
    # may need to be changed if column labels change
    nodata_gdf.drop(columns=['2015_POP_nearest', '2015_NTL_nearest', 'elec_stat_nearest', 'geometry_nearest',
                             'centroid_nearest', '2015_GDP_nearest'], inplace=True)
    
    # returning nodata_gdf with added nearest neighbour estimates
    return nodata_gdf


In [ ]:
for idx, country in representative_countries.iterrows():
    elec_pop_gdp_est_name = country['Code'] + '_elec_pop_gdp_est'
    unelec_pop_gdp_est_name = country['Code'] + '_unelec_pop_gdp_est'
    elec_pop_gdp_data_name = country['Code'] + '_elec_pop_gdp_data'
    unelec_pop_gdp_data_name = country['Code'] + '_unelec_pop_gdp_data'
    elec_pop_gdp_nodata_name = country['Code'] + '_elec_pop_gdp_nodata'
    unelec_pop_gdp_nodata_name = country['Code'] + '_unelec_pop_gdp_nodata'
    
    # finding nearest neighbors and joining data for electrified population
    locals()[elec_pop_gdp_est_name] = nearest_inc(locals()[elec_pop_gdp_nodata_name], locals()[elec_pop_gdp_data_name])
    # calculating total GDP for electrified cells with estimated income levels
    locals()[elec_pop_gdp_est_name]['2015_GDP'] = locals()[elec_pop_gdp_est_name]['2015_INC'] * locals()[elec_pop_gdp_est_name]['2015_POP']
    
    # finding nearest neighbors and joining data for un-electrified population
    locals()[unelec_pop_gdp_est_name] = nearest_inc(locals()[unelec_pop_gdp_nodata_name], locals()[unelec_pop_gdp_data_name])
    # calculating total GDP for un-electrified cells with estimated income levels
    locals()[unelec_pop_gdp_est_name]['2015_GDP'] = locals()[unelec_pop_gdp_est_name]['2015_INC'] * locals()[unelec_pop_gdp_est_name]['2015_POP']
    
# displaying the updated GeoDataFrame
NAM_elec_pop_gdp_est.head()

### 5.4 Re-combining GeoDataFrames and verifying base year GDP against official figures

In [ ]:
for idx, country in representative_countries.iterrows():
    pop_gdp_append_name = country['Code'] + '_pop_gdp_append'
    elec_pop_gdp_est_name = country['Code'] + '_elec_pop_gdp_est'
    unelec_pop_gdp_est_name = country['Code'] + '_unelec_pop_gdp_est'
    elec_pop_gdp_data_name = country['Code'] + '_elec_pop_gdp_data'
    unelec_pop_gdp_data_name = country['Code'] + '_unelec_pop_gdp_data'
    
    # appending the four GeoDataframes
    locals()[pop_gdp_append_name] = pd.concat([locals()[elec_pop_gdp_data_name],
                                locals()[elec_pop_gdp_est_name].drop(columns=['nearest_index']),
                                locals()[unelec_pop_gdp_data_name],
                                locals()[unelec_pop_gdp_est_name].drop(columns=['nearest_index'])])
    
    # re-checking total GDP after applying nearest neighbour estimates
    ssp_gdp = df.loc[df['Country'] == country['Country'], 'GDP_2017USD'].values[0] # official figure from SSP historical reference (OECD ENV-Growth 2023)
    tot_gdp = locals()[pop_gdp_append_name]['2015_GDP'].sum() # GDP total from Kummu et al. + nearest neighbour estimates
    print('Checking total GDP of ' + country['Country'])
    print('Total GDP, SSP:', f"{ssp_gdp:,.0f}", 'USD')
    print('Total GDP, Kummu & nearest neighbour estimates:', f"{tot_gdp:,.0f}", 'USD')
    print('Relative error:', f"{(tot_gdp - ssp_gdp) / ssp_gdp * 100:.2f}", '%')

    # applying a correction factor to match SSP historical reference figures, re-calculating income levels
    corr_fac = ssp_gdp / tot_gdp
    locals()[pop_gdp_append_name]['2015_GDP_corr'] = locals()[pop_gdp_append_name]['2015_GDP'] * corr_fac
    locals()[pop_gdp_append_name]['2015_INC_corr'] = locals()[pop_gdp_append_name]['2015_GDP_corr'] / locals()[pop_gdp_append_name]['2015_POP']
    gdp_corr = locals()[pop_gdp_append_name]['2015_GDP_corr'].sum()
    inc_corr = locals()[pop_gdp_append_name]['2015_GDP_corr'].sum() / locals()[pop_gdp_append_name]['2015_POP'].sum()
    ssp_inc = df.loc[df['Country'] == country['Country'], 'INC_2017USD_calc'].values[0] # official figure from SSP historical reference (OECD ENV-Growth 2023)
    print('Corrected total GDP:', f"{gdp_corr:,.0f}", 'USD')
    print('Resulting income level:', f"{inc_corr:,.0f}", 'USD/capita')
    print('Income level, SSP:', f"{ssp_inc:,.0f}", 'USD/capita')

### 5.5 Saving harmonised base year socioeconomic data to shapefiles

This step saves the harmonised population, GDP, and electrification status data for the base year. Harmonisation was done using SSP historical reference values for total population count and GDP, and World Bank historical values for national electricity access rate.

Harmonised data is used in other scripts as a basis for:
- Base year validation for income distribution and electricity demand estimation methods
- Future projections of socioeconomic indicators and resulting electricity demand in different SSP narratives

In [ ]:
for idx, country in representative_countries.iterrows():
    pop_gdp_append_name = country['Code'] + '_pop_gdp_append'
    gdf_base_name = country['Code'] + '_gdf_base'

    locals()[gdf_base_name] = locals()[pop_gdp_append_name][['geometry', '2015_POP', '2015_NTL', 'elec_stat', '2015_GDP_corr', '2015_INC_corr']]
    #cols_to_float = ['2015_POP', '2015_NTL', '2015_GDP_corr', '2015_INC_corr']
    #locals()[gdf_base_name][cols_to_float] = locals()[gdf_base_name][cols_to_float].astype(float)
    locals()[gdf_base_name].set_geometry('geometry', inplace=True, crs='EPSG:4326')
    locals()[gdf_base_name].to_file('1_output/'+country['Code']+'_socioecon_base_2015_Kummu.gpkg', driver='GPKG')


## End of base year preparation script